# Member C -- Exploration (Epsilon) Experiments

Baseline config comes from `shared_train.py`. Only the exploration
params (`exploration_initial_eps`, `exploration_final_eps`,
`exploration_fraction`) vary across these 10 combos -- everything else
must stay at the shared baseline so results are comparable across the
whole group.

Running on Kaggle. Before running: turn on GPU (Settings -> Accelerator)
and turn on Internet (Settings -> Internet), otherwise the clone and
installs below will fail.

This version pushes results to GitHub after every single run, not just
at the end. If Kaggle kills the session (idle timeout or the ~9-12h
session cap), you may come back to a completely fresh container with
`/kaggle/working` wiped -- pushing after each run means you never lose
more than the one run that was in progress when it died. On restart,
the notebook re-clones the repo, which already has your finished rows,
and the loop below skips any run already in the log.

In [1]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py "autorom[accept-rom-license]" pandas
!AutoROM --accept-license

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 13.9 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/AutoROM/AutoROM.py:264: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


## Get shared_train.py

Clone the group repo so the import below works. Working directory is
`/kaggle/working` here instead of `/content`, since `/content` is a
Colab-only path and does not exist on Kaggle.

In [2]:
import os

REPO_DIR = "/kaggle/working/deep-q-learning-formative3"
if os.path.exists(REPO_DIR):
    # reset --hard (not pull) so this always matches the remote exactly,
    # even if history upstream was rewritten (force-pushed) since your
    # last clone -- a plain pull can't reconcile that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/Mikekimm/deep-q-learning-formative3.git {REPO_DIR}
%cd {REPO_DIR}

from shared_train import BASELINE_CONFIG, GAME_ID, TOTAL_TIMESTEPS, SEED, train_one_run
print(GAME_ID, TOTAL_TIMESTEPS, SEED)
print(BASELINE_CONFIG)

Cloning into '/kaggle/working/deep-q-learning-formative3'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 62 (delta 33), reused 56 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 20.61 KiB | 2.94 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/kaggle/working/deep-q-learning-formative3
ALE/Pong-v5 200000 42
{'learning_rate': 0.0001, 'gamma': 0.99, 'batch_size': 32, 'exploration_initial_eps': 1.0, 'exploration_final_eps': 0.05, 'exploration_fraction': 0.1}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## One-time setup: GitHub push credentials

Before your first run: create a GitHub fine-grained personal access
token scoped to just this repo, with Contents: read and write. In
Kaggle go to Add-ons -> Secrets and add it as `Github_Token`.

Do not paste the token into a cell -- that puts it in the notebook
file. This cell pulls it from Kaggle's secrets manager instead, points
the git remote at an authenticated URL once, then drops the token
variable. `push_results()` below is called after every run.

In [3]:
import subprocess
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("Github_Token")

subprocess.run(["git", "config", "user.email", "memberc@example.com"], cwd=REPO_DIR)
subprocess.run(["git", "config", "user.name", "Member C"], cwd=REPO_DIR)
subprocess.run(
    ["git", "remote", "set-url", "origin",
     f"https://{GITHUB_TOKEN}@github.com/Mikekimm/deep-q-learning-formative3.git"],
    cwd=REPO_DIR,
)
del GITHUB_TOKEN  # remote URL now holds it on disk for this session; no need to keep it in memory

def push_results(commit_msg):
    """Commit and push results/experiments_log.csv. Never raises -- a
    push failure (network blip, race with a teammate's push) should not
    kill a training run that already succeeded."""
    try:
        subprocess.run(["git", "add", "results/experiments_log.csv"], cwd=REPO_DIR, check=True)
        commit = subprocess.run(
            ["git", "commit", "-m", commit_msg], cwd=REPO_DIR,
            capture_output=True, text=True,
        )
        print(commit.stdout.strip() or commit.stderr.strip())

        subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
        rebase = subprocess.run(
            ["git", "rebase", "origin/main"], cwd=REPO_DIR,
            capture_output=True, text=True,
        )
        print(rebase.stdout.strip() or rebase.stderr.strip())

        push = subprocess.run(
            ["git", "push", "origin", "HEAD:main"], cwd=REPO_DIR,
            capture_output=True, text=True,
        )
        print(push.stdout.strip() or push.stderr.strip())
    except Exception as e:
        print("push_results failed, continuing without stopping training:", e)

## Smoke test -- run this ONE cell before anything below

This runs the exact same pipeline as a real experiment, but for only
2,000 timesteps (~1-2 minutes) instead of 200,000 (~hours). It exists
to catch install/setup errors early in *your* session -- Kaggle sessions
can have their own quirks independent of the code. Do NOT use "Run All"
on this notebook -- run this cell by itself first, confirm it prints a
result with no errors, THEN move on to the real experiment cells below.

In [4]:
import shared_train

_original_total_timesteps = shared_train.TOTAL_TIMESTEPS
shared_train.TOTAL_TIMESTEPS = 2000  # temporary override for this test only

smoke_row = shared_train.train_one_run(overrides={}, run_name="smoketest", member="test")
print(smoke_row)

shared_train.TOTAL_TIMESTEPS = _original_total_timesteps  # restore before real runs below
print("TOTAL_TIMESTEPS restored to", shared_train.TOTAL_TIMESTEPS)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /kaggle/working/deep-q-learning-formative3/results/tb_logs/smoketest_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 807      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 128      |
|    time_elapsed     | 6        |
|    total_timesteps  | 780      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0174   |
|    n_updates        | 169      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 785      |
|    ep_rew_mean      | -21      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 147      |
|    time_elapsed     | 10       |
|    total_timestep

In [5]:
# 10 combos of (initial_eps, final_eps, fraction of training spent decaying).
# Each changes one dimension of exploration relative to the baseline so the
# effect stays interpretable.
epsilon_combos = [
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.20, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.10, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.20},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.30},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.01, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 0.8, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.30, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.50},
]

## Optional: watch reward curves live while training runs

Run this once before starting the loop below to open an inline
TensorBoard dashboard -- it updates live as each run trains, so you can
watch reward trends without waiting for a run to finish. (This shows the
reward *graph*, not the game itself -- see the README for why we don't
render pixels during training.) Note: on Kaggle this panel sometimes
stays blank even though logging works fine -- if so, skip it and read
the CSV at the end instead.

In [6]:
%load_ext tensorboard
%tensorboard --logdir results/tb_logs

<IPython.core.display.Javascript object>

## Run experiments (resumable, pushes after each run)

Before starting a run, check whether that run_name is already in
`results/experiments_log.csv`. After each run finishes, push the log to
GitHub immediately -- so if this session dies right after, the row is
already safe and a restart will skip it instead of retraining it.

In [7]:
import csv

def already_done(run_name):
    if not os.path.isfile(shared_train.RESULTS_CSV):
        return False
    with open(shared_train.RESULTS_CSV, newline="") as f:
        reader = csv.DictReader(f)
        return any(r["run_name"] == run_name for r in reader)

for i, combo in enumerate(epsilon_combos, start=1):
    run_name = f"memberC_eps_{i:02d}"
    if already_done(run_name):
        print(f"[{run_name}] already in results log, skipping")
        continue
    train_one_run(
        overrides=combo,
        run_name=run_name,
        member="MemberC",
    )
    push_results(f"Add {run_name} result")

Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /kaggle/working/deep-q-learning-formative3/results/tb_logs/memberC_eps_01_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 896      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.93     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 220      |
|    time_elapsed     | 3        |
|    total_timesteps  | 869      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0202   |
|    n_updates        | 192      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 897      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.86     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 217      |
|    time_elapsed     | 8        |
|    total_tim

From https://github.com/Mikekimm/deep-q-learning-formative3
   bf40bac..2e871f5  main       -> origin/main


Auto-merging results/experiments_log.csv
CONFLICT (content): Merge conflict in results/experiments_log.csv
remote: Permission to Mikekimm/deep-q-learning-formative3.git denied to Mikekimm.
fatal: unable to access 'https://github.com/Mikekimm/deep-q-learning-formative3.git/': The requested URL returned error: 403
Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /kaggle/working/deep-q-learning-formative3/results/tb_logs/memberC_eps_02_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 910      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.921    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 218      |
|    time_elapsed     | 4        |
|    total_timesteps  | 883      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0294   |
|    n_updates        | 195      |
----------------------------------
---------------

From https://github.com/Mikekimm/deep-q-learning-formative3
   2e871f5..c357c9d  main       -> origin/main


fatal: It seems that there is already a rebase-merge directory, and
I wonder if you are in the middle of another rebase.  If that is the
case, please try
	git rebase (--continue | --abort | --skip)
If that is not the case, please
	rm -fr ".git/rebase-merge"
and run me again.  I am stopping in case you still have something
valuable there.
remote: Permission to Mikekimm/deep-q-learning-formative3.git denied to Mikekimm.
fatal: unable to access 'https://github.com/Mikekimm/deep-q-learning-formative3.git/': The requested URL returned error: 403
Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /kaggle/working/deep-q-learning-formative3/results/tb_logs/memberC_eps_04_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 898      |
|    ep_rew_mean      | -20.8    |
|    exploration_rate | 0.959    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 222      |
|    time_elapsed     | 3  

From https://github.com/Mikekimm/deep-q-learning-formative3
   c357c9d..b69a720  main       -> origin/main


fatal: It seems that there is already a rebase-merge directory, and
I wonder if you are in the middle of another rebase.  If that is the
case, please try
	git rebase (--continue | --abort | --skip)
If that is not the case, please
	rm -fr ".git/rebase-merge"
and run me again.  I am stopping in case you still have something
valuable there.
remote: Permission to Mikekimm/deep-q-learning-formative3.git denied to Mikekimm.
fatal: unable to access 'https://github.com/Mikekimm/deep-q-learning-formative3.git/': The requested URL returned error: 403
Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to /kaggle/working/deep-q-learning-formative3/results/tb_logs/memberC_eps_10_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 873      |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.992    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 180      |
|    time_elapsed     | 4  

## Save results

Read the full results log rather than an in-memory list, since a
resumed session will only have this run's rows in memory -- the log
file on disk (and now on GitHub) has all of them, from this run and any
earlier partial run.

In [8]:
import pandas as pd

full_log = pd.read_csv(shared_train.RESULTS_CSV)
results_df = full_log[full_log["run_name"].str.startswith("memberC_eps_")].copy()
results_df.to_csv("/kaggle/working/memberC_epsilon_results.csv", index=False)
push_results("Final MemberC epsilon results")  # safety-net push in case the last per-run push above failed
results_df

interactive rebase in progress; onto 2e871f5
Last command done (1 command done):
   pick 1309946 Add memberC_eps_01 result
No commands remaining.
You are currently editing a commit while rebasing branch 'main' on '2e871f5'.
  (use "git commit --amend" to amend the current commit)
  (use "git rebase --continue" once you are satisfied with your changes)

nothing to commit, working tree clean
fatal: It seems that there is already a rebase-merge directory, and
I wonder if you are in the middle of another rebase.  If that is the
case, please try
	git rebase (--continue | --abort | --skip)
If that is not the case, please
	rm -fr ".git/rebase-merge"
and run me again.  I am stopping in case you still have something
valuable there.
remote: Permission to Mikekimm/deep-q-learning-formative3.git denied to Mikekimm.
fatal: unable to access 'https://github.com/Mikekimm/deep-q-learning-formative3.git/': The requested URL returned error: 403


,run_name,member,timestamp,learning_rate,gamma,batch_size,exploration_initial_eps,exploration_final_eps,exploration_fraction,total_timesteps,mean_reward,std_reward,model_path,notes
13,memberC_eps_01,MemberC,2026-07-13T14:47:24,0.0001,0.99,32.0,1.0,0.20,0.05,200000.0,-17.2,1.166,/kaggle/working/deep-q-learning-formative3/res...,NaN
15,memberC_eps_02,MemberC,2026-07-13T15:05:46,0.0001,0.99,32.0,1.0,0.10,0.05,200000.0,-16.2,1.600,/kaggle/working/deep-q-learning-formative3/res...,NaN
16,memberC_eps_03,MemberC,2026-07-13T15:24:19,0.0001,0.99,32.0,1.0,0.05,0.05,200000.0,-18.0,1.673,/kaggle/working/deep-q-learning-formative3/res...,NaN
17,memberC_eps_04,MemberC,2026-07-13T15:43:31,0.0001,0.99,32.0,1.0,0.05,0.10,200000.0,-17.8,2.135,/kaggle/working/deep-q-learning-formative3/res...,NaN
18,memberC_eps_05,MemberC,2026-07-13T16:03:24,0.0001,0.99,32.0,1.0,0.05,0.20,200000.0,-16.2,2.315,/kaggle/working/deep-q-learning-formative3/res...,NaN
19,memberC_eps_06,MemberC,2026-07-13T16:23:50,0.0001,0.99,32.0,1.0,0.05,0.30,200000.0,-17.2,2.227,/kaggle/working/deep-q-learning-formative3/res...,NaN
20,memberC_eps_07,MemberC,2026-07-13T16:43:50,0.0001,0.99,32.0,1.0,0.01,0.10,200000.0,-17.0,1.265,/kaggle/working/deep-q-learning-formative3/res...,NaN
21,memberC_eps_08,MemberC,2026-07-13T17:03:35,0.0001,0.99,32.0,0.8,0.05,0.10,200000.0,-17.4,1.020,/kaggle/working/deep-q-learning-formative3/res...,NaN
22,memberC_eps_09,MemberC,2026-07-13T17:23:55,0.0001,0.99,32.0,1.0,0.30,0.10,200000.0,-17.0,1.414,/kaggle/working/deep-q-learning-formative3/res...,NaN
23,memberC_eps_10,MemberC,2026-07-13T17:44:07,0.0001,0.99,32.0,1.0,0.05,0.50,200000.0,-18.6,1.020,/kaggle/working/deep-q-learning-formative3/res...,NaN


## Notes

After each run, record what you observed -- did slower decay (larger
fraction) improve final performance at the cost of slower early
learning? Did a high final_eps prevent convergence by never settling
into exploitation? Explain *why*, not just what.